# ViFinQA — Text-to-Pandas codegen trên Kaggle

**Settings:** Accelerator = GPU T4 x2, Internet = On, Add Input → dataset `vifinqa-payload`.

Notebook yêu cầu payload schema v3 (P2.1 dynamic-k/fact-aware) và kiểm SHA-256 trước khi chạy. Nếu payload cũ/thiếu file, notebook dừng ngay thay vì hot-patch âm thầm.

In [ ]:
import glob, json, pathlib
hits = glob.glob("/kaggle/input/**/retrieval.jsonl", recursive=True)
assert hits, "Chưa attach dataset payload - dùng Add Input ở panel phải"
assert len(hits) == 1, f"Có nhiều payload retrieval.jsonl, hãy chỉ attach một dataset: {hits}"
PAYLOAD = str(pathlib.Path(hits[0]).parent)
manifest_path = pathlib.Path(PAYLOAD) / "payload-manifest.json"
assert manifest_path.exists(), ("Payload cũ không có manifest. Chạy lại local: "
                                "python scripts/04_make_kaggle_payload.py rồi re-upload.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("schema_version") == 3, f"Payload schema cũ: {manifest.get('schema_version')}"
print("PAYLOAD =", PAYLOAD, "| files =", len(manifest.get("files", {})))
import torch
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
# Copy nguyên trạng code đã được manifest fingerprint sang working.
import pathlib, shutil
SRC = pathlib.Path(PAYLOAD) / "code"
DST = pathlib.Path("/kaggle/working/code")
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print("code ->", DST)

In [ ]:
%%time
# Giữ trong major version đã kiểm tra; tránh -U lên major mới giữa các lần chạy.
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print("transformers", transformers.__version__, "bitsandbytes", bitsandbytes.__version__)

In [ ]:
%%time
# Smoke P2.1 fact-aware Selection. Runner tự giảm batch 4→2→1 khi CUDA OOM.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_sel14b_factaware_smoke.jsonl --limit 12 \
    --n 1 --temperature 0.7 --k 0 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 4 --time-budget-min 30 --seed 13 --no-resume

In [ ]:
import collections, json
rows = [json.loads(line) for line in open("/kaggle/working/codegen_sel14b_factaware_smoke.jsonl", encoding="utf-8")]
print("rows", len(rows), collections.Counter(r["source"] for r in rows))
print("signatures", {r.get("run_signature", "")[:16] for r in rows})
for r in rows[:8]:
    print(r["id"], r["source"], r["answer"], r["question"][:65])
    print("  ", (r["pandas_query"] or "")[:150].replace("\n", " ; "))

`source=llm_select` chỉ có nghĩa selection đã parse/synthesize/replay được, chưa chứng minh đáp án đúng. Hãy dùng smoke để kiểm runtime/OOM/format và xem `selection_trace`. Full run bên dưới ghi rule baseline trước rồi checkpoint sau mỗi chunk.

In [ ]:
%%time
# Full P2.1 run. Chạy lại cùng cell + cùng output/config sẽ resume theo run_signature.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_sel14b_factaware.jsonl \
    --n 1 --temperature 0.7 --k 0 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 32 --time-budget-min 400 --seed 13

In [ ]:
# QA tối thiểu trước khi download.
import collections, json, math, pathlib
out = pathlib.Path("/kaggle/working/codegen_sel14b_factaware.jsonl")
rows = [json.loads(line) for line in out.open(encoding="utf-8")]
ids = [r["id"] for r in rows]
assert len(rows) == 1012 and len(set(ids)) == 1012, (len(rows), len(set(ids)))
assert all(math.isfinite(float(r["answer"])) for r in rows)
print(collections.Counter(r["source"] for r in rows))
print("selection outcomes", collections.Counter((r.get("selection_trace") or {}).get("outcome", "not_run") for r in rows))
print("rejections", collections.Counter(code for r in rows for code, n in ((r.get("selection_trace") or {}).get("rejection_counts") or {}).items() for _ in range(n)))
print("OK: 1012 unique finite results ->", out)

## Sau khi chạy xong

Tải `codegen_sel14b_factaware.jsonl` về local rồi chạy:
```text
python scripts/05_build_submission.py --retrieval artifacts/retrieval.jsonl --codegen <đường_dẫn>/codegen_sel14b_factaware.jsonl --out-dir artifacts/submission_sel14b_factaware --sub-k 5
```

Resume trong phiên hiện tại dùng lại cùng `--out`. Muốn resume ở phiên Kaggle mới, phải đưa checkpoint cũ vào `/kaggle/working/codegen_sel14b_factaware.jsonl` trước khi chạy. Runner chỉ reuse LLM records có cùng `run_signature`.

`--k 0` không có nghĩa là không dùng table: nó kích hoạt `route.evidence_budget` động (4–12 table). Không đổi cờ này khi resume.

### Troubleshooting

- `missing payload-manifest.json` / hash mismatch: rebuild và re-upload payload; không bypass trong full run.
- CUDA OOM: runner tự giảm batch; nếu vẫn OOM ở batch 1, thử `--batch-size 2`, rồi mới đổi `--k 8` (đổi `k` tạo run signature mới).
- Hết phiên: download checkpoint; ở phiên mới copy nó về đúng `--out` rồi chạy lại cùng config.
- Đổi model/k/n/temperature/max-tokens/llm-mode tạo signature mới và không reuse answer cũ.